In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

import sys
sys.path.append("/home/cyf/wbi/Virginia/code")

import json
import torch
from importlib import reload

from CoarseFlow.training import train, losses
from CoarseFlow.datasets import synthetic_dataset

reload(train)
reload(losses)
reload(synthetic_dataset)

from CoarseFlow.training.train import train_coarse_matching_model
from CoarseFlow.datasets.synthetic_dataset import (
    build_sameShape_loader,
    summarize_manifest,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124
cuda available: True
device count: 1
current device: 0
device name: NVIDIA A100-SXM4-80GB


读取manifest_summary.json
---

In [2]:
# train.ipynb - Cell 2

MANIFEST_SUMMARY_PATH = "/home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/manifest_summary.json"

with open(MANIFEST_SUMMARY_PATH, "r") as f:
    manifest_summary = json.load(f)

for k, v in manifest_summary.items():
    print(k, "->", v)

for name, path in manifest_summary.items():
    print("\n" + "=" * 100)
    print(name)
    summarize_manifest(path)

stage0_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage0_sanity_train/stage0_train_manifest.json
stage0_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage0_sanity_val/stage0_val_manifest.json
stage1_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage1_K5_varSpacing_train/stage1_train_manifest.json
stage1_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage1_K5_varSpacing_val/stage1_val_manifest.json
stage2_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage2_varK_varSpacing_train/stage2_train_manifest.json
stage2_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage2_varK_varSpacing_val/stage2_val_manifest.json
stage3_train -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_datasets/coarseflow_v6/stage3_hard_train/stage3_train_manifest.json
stage3_val -> /home/cyf/wbi/Virginia/code/CoarseFlow/cached_data

定义模型
---


In [3]:
model_config_v6 = dict(
    # =====================================================
    # Core
    # =====================================================
    dim=96,
    radius=(4, 3, 3),
    temperature=0.05,
    use_learned_matching=True,
    matcher_mode="hybrid",

    # feature stride = /8, output control grid = /16
    control_stride=16,
    encoder_stride=8,

    # Important: use 1 refinement iterations from the beginning
    num_refine_iters=1,
    query_chunk_size=512,

    # =====================================================
    # Moving encoder: raw moving -> /8 feature
    # =====================================================
    moving_base_channels=(24, 48, 96),
    moving_num_blocks=(2, 4, 4),
    moving_mlp_ratio=2.0,

    # Important: align moving attention depth with reference attention
    moving_window_attn_layers=6,
    moving_window_size=8,
    moving_attn_num_heads=4,
    moving_slice_fusion_blocks=1,

    # =====================================================
    # Reference encoder: raw ref -> /8 feature
    # =====================================================
    ref_base_channels=(24, 48, 96),
    ref_num_blocks=(2, 4, 4),
    ref_refine_blocks=1,
    ref_mlp_ratio=2.0,

    # Important: 6-layer 3D reference attention
    ref_attn_layers=6,
    ref_attn_num_heads=4,
    ref_attn_window_size=(4, 8, 8),
    ref_attn_mlp_ratio=2.0,

    # =====================================================
    # Embeddings
    # =====================================================
    use_coord_embed=True,
    use_spacing_embed=True,

    # =====================================================
    # Matcher V2
    # =====================================================
    use_offset_encoding=True,
    use_offset_bias=True,
    use_local_cross_attn=True,
    local_attn_temperature=0.20,
    matcher_cross_attn_layers=3,
    matcher_cross_attn_heads=4,
    matcher_ffn_ratio=2.0,
    matcher_attn_drop=0.0,
    matcher_proj_drop=0.0,
    matcher_init_gamma=1e-3,
    
    # =====================================================
    # Residual refinement: OFF for Stage 1-3
    # =====================================================
    use_coord_residual=False,
    residual_type="spatial",
    residual_hidden_dim=256,
    residual_num_blocks=5,
    residual_max_delta=(1.5, 3.0, 3.0),
    residual_use_disp=True,
    residual_use_3d=True,
    residual_detach_coarse=True,
    residual_detach_features=True,
)

In [ ]:
# ============================================================
# Stage 1: K=5 + variable spacing
# ============================================================

train_loader_stage1, _, _ = build_sameShape_loader(
    manifest_summary["stage1_train"],
    batch_size=8,   # 如果显存够，可以改回 5
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage1, _, _ = build_sameShape_loader(
    manifest_summary["stage1_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage1 = dict(model_config_v6)

model_stage1 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage1,
    val_loader=val_loader_stage1,

    save_dir="./checkpoints/coarseflow_v7_stage1_K5_varSpacing_iter3_attn6",

    num_epochs=150,
    lr=5e-5,
    weight_decay=1e-4,
    batch_size=8,
    num_workers=0,
    use_amp=True,

    **model_config_stage1,

    loss_mode="match",

    # 初期仍然以 match distribution 为主
    lambda_match=1.0,
    lambda_match_kl=1.0,
    lambda_match_ce=0.3,

    # 有 3 次 refine 后，coord 可以稍微保留
    lambda_coord=0.05,
    lambda_disp=0.0,

    # 前期不要太强 regularization
    lambda_smooth=0.001,
    lambda_z_spacing=0.001,
    lambda_disp_mag=0.5,
    
    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    resume_path=None,
    resume_optimizer=False,
    resume_best_val_loss=False,
    strict_load=False,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=339
  key=(K,D,H,W)=(5, 30, 256, 256), n=111
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(5, 20, 256, 256), n=71
  key=(K,D,H,W)=(5, 30, 256, 256), n=19
Total parameters    : 2.055 M
Trainable parameters: 2.055 M
Frozen parameters   : 0.000 M
Module                                       Total(M)    Trainable(M)
--------------------------------------------------------------------------------
coord_mlp                                       0.010           0.010
matcher                                         0.278           0.278
moving_encoder                                  0.775           0.775
query_proj                                      0.019           0.019
ref_proj                                        0.019           0.019
reference_encoder                               0.946           0.946
spacing_mlp                                     0.010           0.010
2026-05-20 10:54:43 | [Logger] log file: ./

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:814: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-20 10:54:47 | [Epoch 001] step 0000/57 | loss=5.1733 | match=4.7847 | kl=2.8725 | ce=6.3742 | coord=7.6846 | disp=7.6846 | smooth=3.6240 | z_spacing=0.7625 | top1=0.002 | top5=0.010 | pmax=0.027 | ent=5.533 | inside_valid=1.000 | cand_valid=0.835 | inside_all=1.000 | valid_inside_all=0.667 | time=2.8s | valid=0.667 | 
2026-05-20 10:55:02 | [Epoch 001] step 0010/57 | loss=4.4545 | match=4.1332 | kl=2.3883 | ce=5.8163 | coord=6.3634 | disp=6.3634 | smooth=2.3567 | z_spacing=0.7917 | top1=0.006 | top5=0.031 | pmax=0.008 | ent=5.880 | inside_valid=1.000 | cand_valid=0.840 | inside_all=1.000 | valid_inside_all=0.599 | time=18.1s | valid=0.599 | 
2026-05-20 10:55:17 | [Epoch 001] step 0020/57 | loss=4.1128 | match=3.7956 | kl=2.1384 | ce=5.5240 | coord=6.2556 | disp=6.2556 | smooth=1.9590 | z_spacing=2.4698 | top1=0.022 | top5=0.088 | pmax=0.007 | ent=5.783 | inside_valid=1.000 | cand_valid=0.762 | inside_all=1.000 | valid_inside_all=0.537 | time=32.9s | valid=0.537 | 
2026-05-20 10:

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:340: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-20 10:56:18 | [Val Epoch 001] loss=2.9958 | match=2.7497 | kl=1.4745 | ce=4.2508 | coord=4.8508 | disp=4.8508 | smooth=1.6612 | z_spacing=1.8361 | top1=0.163 | top5=0.385 | pmax=0.053 | ent=4.864 | inside_valid=1.000 | cand_valid=0.801 | inside_all=0.996 | valid_inside_all=0.609 | valid=0.609 | pred_mag=2.162 | gt_mag=2.473
2026-05-20 10:56:20 | [Epoch 002] step 0000/57 | loss=2.7688 | match=2.5673 | kl=1.4034 | ce=3.8798 | coord=3.9810 | disp=3.9810 | smooth=1.6876 | z_spacing=0.6929 | top1=0.237 | top5=0.466 | pmax=0.076 | ent=4.569 | inside_valid=1.000 | cand_valid=0.830 | inside_all=1.000 | valid_inside_all=0.622 | time=1.7s | valid=0.622 | 
2026-05-20 10:56:34 | [Epoch 002] step 0010/57 | loss=2.8461 | match=2.5903 | kl=1.3949 | ce=3.9849 | coord=5.0725 | disp=5.0725 | smooth=1.5017 | z_spacing=0.6483 | top1=0.238 | top5=0.486 | pmax=0.059 | ent=4.707 | inside_valid=1.000 | cand_valid=0.837 | inside_all=1.000 | valid_inside_all=0.657 | time=15.5s | valid=0.657 | 
2026-05-2

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Stage 2: variable K + variable spacing
# ============================================================

train_loader_stage2, _, _ = build_sameShape_loader(
    manifest_summary["stage2_train"],
    batch_size=7,   # 如果显存够，可以改成 4
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage2, _, _ = build_sameShape_loader(
    manifest_summary["stage2_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage2 = dict(model_config_v6)


model_stage2 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage2,
    val_loader=val_loader_stage2,

    save_dir="./checkpoints/coarseflow_v7_stage2_varK_varSpacing_iter3_attn6",

    num_epochs=300,
    lr=3e-5,
    weight_decay=1e-4,
    batch_size=7,
    num_workers=0,
    use_amp=True,

    **model_config_stage2,

    loss_mode="match",

    lambda_match=1.0,
    lambda_match_kl=1.0,
    lambda_match_ce=0.3,

    lambda_coord=0.1,
    lambda_disp=0.0,
    lambda_smooth=0.001,
    lambda_z_spacing=0.001,

    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    resume_path="./checkpoints/coarseflow_v7_stage1_K5_varSpacing_iter3_attn6/best.pth",
    resume_optimizer=False,
    resume_best_val_loss=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="w",
)

[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=230
  key=(K,D,H,W)=(3, 30, 256, 256), n=70
  key=(K,D,H,W)=(4, 20, 256, 256), n=235
  key=(K,D,H,W)=(4, 30, 256, 256), n=65
  key=(K,D,H,W)=(5, 20, 256, 256), n=217
  key=(K,D,H,W)=(5, 30, 256, 256), n=83
  key=(K,D,H,W)=(6, 20, 256, 256), n=223
  key=(K,D,H,W)=(6, 30, 256, 256), n=77
  key=(K,D,H,W)=(7, 20, 256, 256), n=224
  key=(K,D,H,W)=(7, 30, 256, 256), n=76
[SameShapeBatchSampler] groups:
  key=(K,D,H,W)=(3, 20, 256, 256), n=51
  key=(K,D,H,W)=(3, 30, 256, 256), n=21
  key=(K,D,H,W)=(5, 20, 256, 256), n=60
  key=(K,D,H,W)=(5, 30, 256, 256), n=12
  key=(K,D,H,W)=(7, 20, 256, 256), n=62
  key=(K,D,H,W)=(7, 30, 256, 256), n=10
Total parameters    : 2.055 M
Trainable parameters: 2.055 M
Frozen parameters   : 0.000 M
Module                                       Total(M)    Trainable(M)
--------------------------------------------------------------------------------
coord_mlp                                       0.

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:814: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)
/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-20 13:50:05 | [Epoch 096] step 0000/216 | loss=2.1915 | match=1.7968 | kl=0.9292 | ce=2.8918 | coord=3.9324 | disp=3.9324 | smooth=0.9348 | z_spacing=0.6012 | top1=0.238 | top5=0.793 | pmax=0.139 | ent=3.886 | inside_valid=1.000 | cand_valid=0.774 | inside_all=1.000 | valid_inside_all=0.636 | time=2.3s | valid=0.636 | 
2026-05-20 13:50:19 | [Epoch 096] step 0010/216 | loss=2.3299 | match=1.8776 | kl=1.0291 | ce=2.8285 | coord=4.4892 | disp=4.4892 | smooth=0.6931 | z_spacing=2.6810 | top1=0.361 | top5=0.745 | pmax=0.174 | ent=3.717 | inside_valid=1.000 | cand_valid=0.747 | inside_all=0.999 | valid_inside_all=0.672 | time=16.4s | valid=0.672 | 
2026-05-20 13:50:33 | [Epoch 096] step 0020/216 | loss=2.4751 | match=2.0335 | kl=1.0634 | ce=3.2337 | coord=4.4034 | disp=4.4034 | smooth=1.0752 | z_spacing=0.1416 | top1=0.347 | top5=0.638 | pmax=0.118 | ent=4.089 | inside_valid=1.000 | cand_valid=0.823 | inside_all=1.000 | valid_inside_all=0.625 | time=29.8s | valid=0.625 | 
2026-05-20 

/home/cyf/wbi/Virginia/code/CoarseFlow/training/train.py:340: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


2026-05-20 13:55:10 | [Val Epoch 096] loss=2.0904 | match=1.7089 | kl=0.8285 | ce=2.9345 | coord=3.7916 | disp=3.7916 | smooth=0.9071 | z_spacing=1.5068 | top1=0.365 | top5=0.829 | pmax=0.106 | ent=4.150 | inside_valid=1.000 | cand_valid=0.793 | inside_all=0.997 | valid_inside_all=0.608 | valid=0.608 | pred_mag=1.009 | gt_mag=2.582
2026-05-20 13:55:11 | [Epoch 097] step 0000/216 | loss=2.2231 | match=1.7978 | kl=0.8901 | ce=3.0254 | coord=4.2411 | disp=4.2411 | smooth=1.0864 | z_spacing=0.1345 | top1=0.368 | top5=0.772 | pmax=0.102 | ent=4.193 | inside_valid=1.000 | cand_valid=0.827 | inside_all=1.000 | valid_inside_all=0.616 | time=1.4s | valid=0.616 | 
2026-05-20 13:55:24 | [Epoch 097] step 0010/216 | loss=2.4631 | match=2.0240 | kl=1.1314 | ce=2.9753 | coord=4.3588 | disp=4.3588 | smooth=0.6703 | z_spacing=2.5894 | top1=0.380 | top5=0.687 | pmax=0.176 | ent=3.678 | inside_valid=1.000 | cand_valid=0.715 | inside_all=0.993 | valid_inside_all=0.570 | time=14.2s | valid=0.570 | 
2026-05

In [2]:
# ============================================================
# Stage 3: sharpen / fine-tune with iter3
# ============================================================

train_loader_stage3, _, _ = build_sameShape_loader(
    manifest_summary["stage3_train"],
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

val_loader_stage3, _, _ = build_sameShape_loader(
    manifest_summary["stage3_val"],
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

model_config_stage3 = dict(model_config_v6)
model_config_stage3.update(
    dict(
        num_refine_iters=3,
        moving_window_attn_layers=6,
        ref_attn_layers=6,
        use_coord_residual=False,
    )
)

model_stage3 = train_coarse_matching_model(
    train_dataset=None,
    val_dataset=None,
    train_loader=train_loader_stage3,
    val_loader=val_loader_stage3,

    save_dir="./checkpoints/coarseflow_v6_stage3_iter3_attn6_sharpen",

    num_epochs=120,
    lr=5e-6,
    weight_decay=1e-4,
    batch_size=2,
    num_workers=0,
    use_amp=True,

    **model_config_stage3,

    loss_mode="match",

    # Sharpen distribution
    lambda_match=1.0,
    lambda_match_kl=0.25,
    lambda_match_ce=0.45,

    # More coordinate supervision
    lambda_coord=0.3,
    lambda_disp=0.0,

    lambda_smooth=0.001,
    lambda_z_spacing=0.001,

    compute_chunk_match_loss=True,
    match_sigma=(0.5, 1.0, 1.0),
    match_inside_threshold=4.0,

    resume_path="./checkpoints/coarseflow_v7_stage2_varK_varSpacing_iter3_attn6/best.pth",
    resume_optimizer=False,
    resume_best_val_loss=False,
    strict_load=True,

    log_filename="train.log",
    log_mode="w",
)

NameError: name 'manifest_summary' is not defined